In [1]:
# from huggingface_hub import hf_hub_download

# hf_hub_download(
#     repo_id="malaysia-ai/pseudolabel-malaysia-parliament-youtube-whisper-large-v3",
#     repo_type="dataset",
#     filename="parlimen-force-alignment.zip",
#     local_dir="./"
# )

In [2]:
from multiprocess import Pool
import itertools
import json
from datasets import load_dataset

def chunks(l, n):
    for i in range(0, len(l), n):
        yield (l[i: i + n], i // n)

def multiprocessing(strings, function, cores=6, returned=True):
    df_split = chunks(strings, len(strings) // cores)
    pool = Pool(cores)
    pooled = pool.map(function, df_split)
    pool.close()
    pool.join()

    if returned:
        return list(itertools.chain(*pooled))

/home/ubuntu/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
ds = load_dataset("malaysia-ai/pseudolabel-malaysia-parliament-youtube-whisper-large-v3")

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"
Generating train split: 100%|██████████| 479134/479134 [00:00<00:00, 2231711.24 examples/s]


In [4]:
df = ds['train'].to_pandas().to_dict(orient = 'records')

In [5]:
df = [(i, df[i]) for i in range(len(df))]
df[0]

(0,
 {'audio_filename': 'chunk-30s-parlimen/1321-45.mp3',
  'text': 'Dalam kenyataan Bank Negara itu mengesahkan bahawa kesemua pelaburan di luar negara oleh entiti Pemastautin tertakluk kepada Akta Kawalan Pertukaran Wang. Kemudian disebut bahawa Bank Negara telah memberikan maklumat kepada Agensi Penguatkuasaan Undang-Undang yang berkaitan pada bulan April 2016.'})

In [6]:
# if between word distances > 2
# if word distance > 1
# if word[0] start > 5
# if len sentence > 600

from tqdm import tqdm

def loop(rows):
    rows, _ = rows
    selected = []
    for r in tqdm(rows):
        if len(r[1]['text']) > 600:
            continue

        try:
            with open(f'force-alignment/{r[0]}.json') as fopen:
                d = json.load(fopen)
        except:
            continue

        if d[0]['start'] > 5:
            continue

        failed = False
        for i in range(len(d)):
            if i > 0 and (d[i]['start'] - d[i - 1]['end']) > 2:
                failed = True
                break

            if (d[i]['end'] - d[i]['start']) > 1.:
                failed = True
                break

        if failed:
            continue

        selected.append((r[1], d))
    return selected

In [7]:
filtered = loop((df[:100], 0))

100%|██████████| 100/100 [00:00<00:00, 22392.31it/s]


In [8]:
filtered = multiprocessing(df, loop, cores = 30)

100%|██████████| 15971/15971 [00:00<00:00, 28194.22it/s]


In [9]:
len(filtered) / len(df), len(filtered), len(df)

(0.10286892602069567, 49288, 479134)

In [10]:
import IPython.display as ipd
ipd.Audio('chunk-30s-parlimen/450-24.mp3')

In [11]:
from collections import defaultdict
import os

audio_names = defaultdict(list)
for r in tqdm(filtered):
    audio_names[os.path.split(r[0]['audio_filename'])[1].split('-')[0]].append(r)

100%|██████████| 49288/49288 [00:00<00:00, 831699.36it/s]


In [12]:
# streaming base
## segment level
## word level
# whole chunk base
## segment level
## word level

In [28]:
keys = list(audio_names.keys())

group = []
for k in tqdm(keys):
    s = sorted(audio_names[k], key = lambda x: int(os.path.split(x[0]['audio_filename'])[1].split('-')[1].replace('.mp3', '')))
    temp = [s[0]]
    previous = int(s[0][0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
    final = False
    for s_ in s[1:]:
        i = int(s_[0]['audio_filename'].split('-')[-1].replace('.mp3', ''))
        if previous + 1 == i:
            final = False
            temp.append(s_)
            previous += 1
        else:
            final = True
            group.append(temp)
            temp = [s_]
            previous = i
    
    if not final:
        group.append(temp)

100%|██████████| 1808/1808 [00:00<00:00, 6143.70it/s]


In [29]:
group = [(i, group[i]) for i in range(len(group))]

In [57]:
!rm -rf parlimen-whole parlimen-segment
!mkdir parlimen-whole
!mkdir parlimen-segment

In [58]:
len(group)

39877

In [59]:
import copy
import soundfile as sf
import librosa
import numpy as np

def loop(group):
    group, _ = group
    combine_all = []
    for g in tqdm(group):
        i = g[0]
        g = g[1]
        audio_files = []
        timestamps = []
        last_timestamp = 0
        for g_ in g:
            audio_files.append(g_[0]['audio_filename'])
            timestamp = copy.deepcopy(g_[1])
            for k in range(len(timestamp)):
                timestamp[k]['start'] += last_timestamp
                timestamp[k]['end'] += last_timestamp
            timestamps.extend(timestamp)
            last_timestamp = timestamp[-1]['end']
    
        word_level = []
        for t in timestamps:
            start = t['start']
            w = t['text']
            end = t['end']
            word_level.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
        
        segments, temp = [], [timestamps[0]]
        last_t = timestamps[0]['end']
        for c_ in timestamps[1:]:
            if ((c_['start'] - last_t) > 0.4):
                segments.append(temp)
                temp = []
    
            last_t = c_['end']
            temp.append(c_)
    
        if len(temp):
            segments.append(temp)
    
        segment_level = []
        for s in segments:
            start = s[0]['start']
            end = s[-1]['end']
            w = ' '.join([c_['text'] for c_ in s])
            t = f"<|{start:.2f}|> {w}<|{end:.2f}|>"
            segment_level.append(t)
    
        y = [librosa.load(f, sr = 16000)[0] for f in audio_files]
        y = np.concatenate(y)
    
        audio_filename = f'parlimen-whole/{i}.mp3'
        sf.write(audio_filename, y, 16000)
    
        segment_audio_filenames = []
        streaming_word_level = []
        for k, s in enumerate(segments):
            segment_audio_filename = f'parlimen-segment/{i}-{k}.mp3'
            start = s[0]['start']
            end = s[-1]['end']
            y_ = y[int(start * 16000): int(end * 16000)]
            sf.write(segment_audio_filename, y_, 16000)
            segment_audio_filenames.append(segment_audio_filename)
    
            word_level_ = []
            for t in s:
                start = t['start']
                w = t['text']
                end = t['end']
                word_level_.append(f"<|{start:.2f}|> {w}<|{end:.2f}|>")
            streaming_word_level.append(''.join(word_level_))
            
        word_level = ''.join(word_level)
    
        combine_all.append({
            'mode': 'whole',
            'level': 'segment',
            'texts': [''.join(segment_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'whole',
            'level': 'word',
            'texts': [''.join(word_level)],
            'audio_filenames': [audio_filename],
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'segment',
            'texts': segment_level,
            'audio_filenames': segment_audio_filenames,
        })
        combine_all.append({
            'mode': 'streaming',
            'level': 'word',
            'texts': streaming_word_level,
            'audio_filenames': segment_audio_filenames,
        })
    return combine_all

In [60]:
combine_all = loop((group[:2], 0))

100%|██████████| 2/2 [00:00<00:00,  2.86it/s]


In [63]:
combine_all[-1]

{'mode': 'streaming',
 'level': 'word',
 'texts': ['<|0.04|> Pagi<|0.14|><|0.24|> ini<|0.38|>',
  '<|1.30|> adalah<|1.64|><|1.68|> merupakan<|2.24|>',
  '<|4.16|> yang<|4.36|>',
  '<|4.88|> berhormat<|5.26|>',
  '<|5.76|> Datuk<|6.02|><|6.06|> Indra<|6.28|>',
  '<|6.86|> Mahmoud<|7.12|><|7.20|> Sahr<|7.46|><|7.70|> bin<|7.86|><|8.16|> Abdullah<|8.54|><|8.70|> yang<|8.76|><|8.80|> merupakan<|9.28|><|9.42|> ahli<|9.60|>',
  '<|10.26|> Parlimen<|10.68|><|10.80|> Paya<|10.98|><|11.04|> Besar<|11.34|><|11.50|> merangkap<|11.94|><|12.16|> pengurusi<|12.60|>',
  '<|13.40|> jadualan<|13.74|><|13.82|> puasa<|14.12|><|14.20|> pilihan<|14.46|><|14.58|> khas<|14.82|><|15.02|> kewangan<|15.46|><|15.52|> dan<|15.72|>',
  '<|16.14|> ekonomi.<|16.56|>',
  '<|17.32|> Kemudian<|17.76|><|18.04|> di<|18.10|><|18.18|> sebelah<|18.46|>',
  '<|19.24|> kiri<|19.44|><|19.74|> pada<|19.94|>',
  '<|21.00|> sesi<|21.24|><|21.56|> pagi<|21.76|><|21.82|> ini<|21.90|><|22.04|> ialah<|22.26|><|22.38|> yang<|22.50|><|

In [67]:
ipd.Audio(combine_all[-1]['audio_filenames'][-1])

In [68]:
combine_all = multiprocessing(group, loop, cores = 50)

100%|██████████| 797/797 [14:00<00:00,  1.05s/it]


In [72]:
len(combine_all)

159508

In [73]:
import pandas as pd

pd.DataFrame(combine_all).to_parquet('new-parlimen.parquet')

In [74]:
from datasets import Dataset

dataset = Dataset.from_list(combine_all)

In [76]:
dataset.push_to_hub('malaysia-ai/Malaysian-STT', 'parliament')

Uploading the dataset shards: 100%|██████████| 1/1 [00:14<00:00, 14.43s/it]


CommitInfo(commit_url='https://huggingface.co/datasets/malaysia-ai/Malaysian-STT/commit/f2651da2cb36614dec1944a3be2d1531ed306b70', commit_message='Upload dataset', commit_description='', oid='f2651da2cb36614dec1944a3be2d1531ed306b70', pr_url=None, repo_url=RepoUrl('https://huggingface.co/datasets/malaysia-ai/Malaysian-STT', endpoint='https://huggingface.co', repo_type='dataset', repo_id='malaysia-ai/Malaysian-STT'), pr_revision=None, pr_num=None)

In [77]:
!du -hs parlimen-whole

7.4G	parlimen-whole


In [78]:
!du -hs parlimen-segment

7.0G	parlimen-segment


In [83]:
!zip -rq parlimen-whole.zip parlimen-whole

In [ ]:
!zip -rq parlimen-segment.zip parlimen-segment